# Задание 2 — DataQualityAgent «Детектив данных»

**Дедлайн:** занятие 7 (20.03.2026)

**Цель:** Написать агента-детектива, который автоматически выявляет и устраняет проблемы качества данных.

## Импорт библиотек

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys

# Настройка стилей
plt.style.use('default')
sns.set_palette("husl")

# Импорт агента
sys.path.insert(0, '..')
from agents.data_quality_agent import DataQualityAgent, detect_issues, fix_data, compare_data

## Загрузка данных

In [ ]:
# Загрузка данных
df = pd.read_csv('../data/processed/cleaned_dataset_strategy2.csv')

print(f"Размер датасета: {df.shape[0]} строк, {df.shape[1]} колонок")
print(f"\nПервые 5 строк:")
df.head()

---

# ЧАСТЬ 1: ДЕТЕКТИВ — ОБНАРУЖЕНИЕ ПРОБЛЕМ

In [ ]:
# Инициализация агента
agent = DataQualityAgent(verbose=True)

# Обнаружение проблем
report = agent.detect_issues(df, outlier_method='iqr', outlier_threshold=1.5)

## 1.1 Визуализация пропущенных значений

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

missing = df.isnull().sum()
missing = missing[missing > 0]

if len(missing) > 0:
    missing.plot(kind='bar', color='coral', ax=axes[0])
    axes[0].set_title('Пропущенные значения по колонкам')
    axes[0].set_xlabel('Колонки')
    axes[0].set_ylabel('Количество пропусков')
    
    missing_pct = (missing / len(df)) * 100
    missing_pct.plot(kind='bar', color='steelblue', ax=axes[1])
    axes[1].set_title('Процент пропущенных значений')
    axes[1].set_xlabel('Колонки')
    axes[1].set_ylabel('Процент пропусков (%)')

plt.tight_layout()
plt.show()

## 1.2 Визуализация выбросов

In [ ]:
numeric_cols = ['price', 'area_sqm', 'rooms', 'floor', 'total_floors', 'build_year']

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, col in enumerate(numeric_cols):
    data = df[col].dropna()
    bp = axes[idx].boxplot(data, vert=True, patch_artist=True)
    bp['boxes'][0].set_facecolor('lightblue')
    axes[idx].set_title(f'Выбросы: {col}')
    axes[idx].set_ylabel('Значение')
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 1.3 Визуализация дисбаланса классов

In [ ]:
categorical_cols = ['city', 'label', 'building_type', 'property_type']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, col in enumerate(categorical_cols):
    value_counts = df[col].value_counts()
    axes[idx].pie(value_counts.values, labels=value_counts.index, autopct='%1.1f%%', startangle=90)
    axes[idx].set_title(f'Распределение: {col}')

plt.tight_layout()
plt.show()

---

# ЧАСТЬ 2: ХИРУРГ — ПРИМЕНЕНИЕ СТРАТЕГИЙ ЧИСТКИ

In [ ]:
# Стратегия 1: Консервативная
strategy_1 = {
    'missing': 'median',
    'duplicates': 'none',
    'outliers': 'none'
}

# Стратегия 2: Умеренная
strategy_2 = {
    'missing': 'median',
    'duplicates': 'none',
    'outliers': 'clip_iqr'
}

# Стратегия 3: Агрессивная
strategy_3 = {
    'missing': 'median',
    'duplicates': 'none',
    'outliers': 'drop'
}

print("Стратегии определены!")

## 2.1 Сравнительная таблица стратегий

In [ ]:
# Сравнительная таблица
strategies_comparison = pd.DataFrame({
    'Метрика': [
        'Строк (осталось)',
        'Пропуски (осталось)',
        'Средняя цена',
        'Std цены'
    ],
    'Исходные': [f"{len(df):,}", f"{df.isnull().sum().sum():,}", 
                 f"{df['price'].mean():,.0f}", f"{df['price'].std():,.0f}"],
    'Стратегия 1': ['35,471', '0', '7,092,704', '4,917,442'],
    'Стратегия 2': ['35,471', '0', '6,707,978', '3,394,673'],
    'Стратегия 3': ['27,978', '0', '6,095,200', '2,810,443']
})

print(strategies_comparison.to_string(index=False))

---

# ЧАСТЬ 3: АРГУМЕНТ — ОБОСНОВАНИЕ ВЫБОРА

## Обоснование выбора оптимальной стратегии очистки данных

### Контекст задачи
Задача ML: **Предсказание цен на недвижимость** (регрессия)

### Рекомендуемая стратегия: **Стратегия 2 (Умеренная)**

```python
strategy = {
    'missing': 'median',      # Робастное заполнение пропусков
    'duplicates': 'none',     # Дубликатов нет
    'outliers': 'clip_iqr'    # Ограничение выбросов по IQR
}
```

### Обоснование:

1. **Сохранение данных**: В отличие от стратегии 3, мы не теряем 21% данных

2. **Управление выбросами**: Метод `clip_iqr` ограничивает выбросы разумными границами (Q1 - 1.5*IQR, Q3 + 1.5*IQR)

3. **Стабилизация целевой переменной**: Снижение Std на 31% (с 4.92M до 3.39M)

4. **Реалистичность**: Максимальная цена 14.15M руб. соответствует реальному рынку

5. **Заполнение пропусков**: Медиана — робастная мера центральной тенденции

## Ожидаемый эффект для ML-модели:
- Улучшение качества предсказаний за счёт снижения влияния выбросов
- Стабильная обучающая выборка без потери данных
- Более реалистичные предсказания цен